# Deep Fusion Ablation Plan

This notebook defines the large-model search space to apply after the fast ML lab. Keep the grid small at first so each result is meaningful.


In [ ]:
import itertools
import pandas as pd

base_models = [
    {'model': 'A', 'backbone': 'EfficientNet-B0', 'temporal': 'GRU'},
    {'model': 'B', 'backbone': 'MobileNetV2', 'temporal': 'LSTM'},
    {'model': 'C', 'backbone': 'EfficientNet-B0', 'temporal': 'LSTM'},
    {'model': 'D', 'backbone': 'MobileNetV2', 'temporal': 'GRU'},
]

configs = []
for base, seq_len, fusion, pooling, loss in itertools.product(
    base_models,
    [5, 8, 10],
    ['early_concat', 'late_env_branch', 'gated_env_branch'],
    ['last', 'last_mean_max'],
    ['mae', 'smooth_l1'],
):
    configs.append({
        **base,
        'seq_len': seq_len,
        'fusion': fusion,
        'temporal_pooling': pooling,
        'loss': loss,
        'dropout': 0.35,
        'scheduler': 'ReduceLROnPlateau',
        'early_stopping_patience': 4,
        'augmentation': True,
    })

ablation_grid = pd.DataFrame(configs)
ablation_grid.head(20)


In [ ]:
# Practical first batch: do not run the whole grid.
first_batch = ablation_grid[
    (ablation_grid['model'] == 'C') &
    (ablation_grid['seq_len'] == 10) &
    (ablation_grid['loss'] == 'smooth_l1') &
    (ablation_grid['fusion'].isin(['early_concat', 'late_env_branch'])) &
    (ablation_grid['temporal_pooling'].isin(['last', 'last_mean_max']))
]
first_batch


## Transfer Rule

Only port a setting to all A/B/C/D after it improves Model C against the reported `32.02h` test MAE or improves validation stability without harming test MAE. This keeps the larger retrain budget focused.
